# T01 — scvi-tools: Deep Generative Models for Single-Cell Data

**Tools:** `scvi-tools` (scVI, scANVI)  
**Dataset:** PBMC 3k (with simulated batches) + Heart Cell Atlas (subsampled)  
**Papers:** [Lopez et al. 2018, Nature Methods](https://doi.org/10.1038/s41592-018-0229-2) · [Xu et al. 2021, Molecular Systems Biology](https://doi.org/10.15252/msb.20209620)

---

## Why variational autoencoders for scRNA-seq?

Library-size normalization + log1p (T00) is fast but ignores key structure:
- **Overdispersion**: scRNA-seq counts are negative-binomially distributed, not normally distributed after log1p
- **Zero inflation**: ~90% of values are zero; many are *technical* (not biological) zeros
- **Batch effects**: systematic technical differences between samples, sequencing runs, or labs

scVI (**single-cell Variational Inference**) trains a variational autoencoder where:
- The **encoder** maps each cell to a low-dimensional latent space `z` (typically 10-20 dims)
- The **decoder** reconstructs expected counts from `z` using a **negative binomial likelihood** with learned dispersion
- A separate linear term captures library size, and optionally a batch covariate

The latent space `z` is the "batch-corrected" representation you use for downstream analysis.

```
  cell (raw counts) → encoder → z (10-20 dim) → decoder → reconstructed NB params
                                     ↑
                              batch covariate goes here
```

scVI is the base model. The ecosystem includes:
| Model | What it does |
|-------|--------------|
| `scVI` | Unsupervised integration / denoising |
| `scANVI` | Semi-supervised: uses known cell type labels to improve `z` |
| `totalVI` | Joint model for CITE-seq (RNA + surface protein) |
| `PEAKVI` | scATAC-seq |
| `multVI` | Multiome (RNA + ATAC) |
| `TOTALVI` | CITE-seq |
| `AutoZI` | Zero-inflated extension |

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import torch
import matplotlib.pyplot as plt

sc.settings.set_figure_params(dpi=80, facecolor='white')
scvi.settings.seed = 42
print(f'scvi-tools {scvi.__version__}  |  torch {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')

## 1. scVI on a Single Dataset

First, the simplest use case: learning a latent representation of one dataset (no batch correction). This is useful for denoising and getting a better low-dimensional embedding than PCA alone.

In [ ]:
# Load processed PBMC 3k from T00 (or regenerate from scratch)
import os
if os.path.exists('../data/pbmc3k_processed.h5ad'):
    adata = sc.read_h5ad('../data/pbmc3k_processed.h5ad')
else:
    # Rebuild from scratch if T00 wasn't run
    adata = sc.datasets.pbmc3k()
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
    adata = adata[adata.obs.n_genes_by_counts < 2500]
    adata = adata[adata.obs.pct_counts_mt < 5]
    adata.layers['counts'] = adata.X.copy()  # preserve raw counts
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)

print(adata)
# scVI requires raw integer counts — confirm the 'counts' layer exists
assert 'counts' in adata.layers, "Need raw counts layer; re-run preprocessing"

In [ ]:
# Setup AnnData for scVI
# This registers which layer to use, which obs columns are covariates, etc.
# Must be called before model instantiation.
scvi.model.SCVI.setup_anndata(
    adata,
    layer='counts',                    # raw counts (not normalized!)
    batch_key=None,                    # no batch for now
)

In [ ]:
# Instantiate and train the model
model = scvi.model.SCVI(
    adata,
    n_layers=2,         # depth of encoder/decoder networks
    n_latent=10,        # dimensionality of z
    gene_likelihood='nb',  # negative binomial (default); 'zinb' for zero-inflated
)
print(model)
model.train(max_epochs=100, early_stopping=True)

In [ ]:
# Training loss curve
train_elbo = model.history['elbo_train']
val_elbo   = model.history['elbo_validation']
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(train_elbo.index, train_elbo.values, label='Train ELBO')
ax.plot(val_elbo.index,   val_elbo.values,   label='Validation ELBO')
ax.set_xlabel('Epoch')
ax.set_ylabel('ELBO')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Extract the latent representation (n_cells × n_latent)
adata.obsm['X_scVI'] = model.get_latent_representation()

# Use the scVI latent space as the input to neighbors + UMAP
# instead of the PCA embedding
sc.pp.neighbors(adata, use_rep='X_scVI', n_neighbors=15)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5)

sc.pl.umap(
    adata,
    color=['leiden', 'cell_type'] if 'cell_type' in adata.obs else ['leiden'],
    legend_loc='on data', title='scVI latent space'
)

In [ ]:
# scVI can also give denoised/normalized expression estimates
# (integrated over the posterior, handling zero-inflation)
denoised = model.get_normalized_expression(library_size=1e4)
print('Denoised expression:', denoised.shape)  # cells × genes
# These are better for visualization than log1p-normalized values

---
## 2. Batch Integration with scVI

The primary use case: you have cells from multiple batches (samples, experiments, technologies) that you want to integrate. scVI handles this by passing `batch_key` so the decoder learns batch-specific reconstruction while the latent `z` is forced to be batch-agnostic.

We'll demonstrate with the **Heart Cell Atlas** subsampled dataset (multiple batches built in).

In [ ]:
# Heart Cell Atlas — 3,000 cells, multiple donors/batches
# This is a scvi-tools built-in dataset
adata_hca = scvi.data.heart_cell_atlas_subsampled()
print(adata_hca)
print('\nBatch counts:')
print(adata_hca.obs['batch'].value_counts())

In [ ]:
# Preprocessing (same pipeline as T00)
sc.pp.filter_genes(adata_hca, min_cells=10)
adata_hca.layers['counts'] = adata_hca.X.copy()
sc.pp.normalize_total(adata_hca, target_sum=1e4)
sc.pp.log1p(adata_hca)
sc.pp.highly_variable_genes(adata_hca, n_top_genes=2000, subset=True, batch_key='batch')
print(adata_hca)

In [ ]:
# Visualize BEFORE integration (scanpy's PCA-based UMAP)
# Expect to see cells segregated by batch
sc.tl.pca(adata_hca)
sc.pp.neighbors(adata_hca)
sc.tl.umap(adata_hca)
sc.pl.umap(adata_hca, color='batch', title='Before integration (PCA)')

In [ ]:
# Now train scVI WITH batch_key — the key parameter for integration
scvi.model.SCVI.setup_anndata(
    adata_hca,
    layer='counts',
    batch_key='batch',                 # <-- tells scVI to correct for this
)

model_hca = scvi.model.SCVI(
    adata_hca,
    n_layers=2,
    n_latent=20,
    gene_likelihood='nb',
)
model_hca.train(max_epochs=150, early_stopping=True)

In [ ]:
# Integrated latent space
adata_hca.obsm['X_scVI'] = model_hca.get_latent_representation()
sc.pp.neighbors(adata_hca, use_rep='X_scVI')
sc.tl.umap(adata_hca)
sc.tl.leiden(adata_hca, resolution=0.5)

sc.pl.umap(
    adata_hca,
    color=['batch', 'leiden', 'cell_type'],
    ncols=3,
    title=['Batch (integrated)', 'Leiden clusters', 'Cell type (original labels)'],
)

---
## 3. scANVI: Semi-Supervised Label Transfer

scANVI extends scVI with a classifier in the latent space. If you have:
- A **labeled reference** dataset (or a subset of labeled cells)
- An **unlabeled query** dataset you want to annotate

scANVI will map both to the same latent space, using the labels as a supervision signal.

A common workflow:
1. Train scVI first (fast, unsupervised)
2. Initialize scANVI from the scVI model
3. scANVI fine-tunes with the labels

In [ ]:
# Initialize scANVI from the trained scVI model
# cells with unknown labels get the string 'Unknown'
# (set some cells to Unknown to simulate partial labeling)
import numpy as np
adata_hca.obs['cell_type_query'] = adata_hca.obs['cell_type'].astype(str)
np.random.seed(42)
unknown_mask = np.random.rand(len(adata_hca)) < 0.3  # 30% unknown
adata_hca.obs.loc[unknown_mask, 'cell_type_query'] = 'Unknown'

scvi.model.SCANVI.setup_anndata(
    adata_hca,
    layer='counts',
    batch_key='batch',
    labels_key='cell_type_query',
    unlabeled_category='Unknown',
)

In [ ]:
# Initialize from scVI weights, then fine-tune
model_scanvi = scvi.model.SCANVI.from_scvi_model(
    model_hca,
    unlabeled_category='Unknown',
)
model_scanvi.train(max_epochs=20)

# Predict labels for unlabeled cells
adata_hca.obs['predicted_cell_type'] = model_scanvi.predict()

# Compare predicted vs actual for the cells we masked
masked = adata_hca[unknown_mask]
correct = (masked.obs['predicted_cell_type'] == masked.obs['cell_type']).mean()
print(f'Accuracy on masked cells: {correct:.1%}')

In [ ]:
adata_hca.obsm['X_scANVI'] = model_scanvi.get_latent_representation()
sc.pp.neighbors(adata_hca, use_rep='X_scANVI')
sc.tl.umap(adata_hca)

sc.pl.umap(
    adata_hca,
    color=['predicted_cell_type', 'cell_type'],
    title=['scANVI predicted', 'True labels'],
)

---
## Summary

| scvi-tools model | Input | Key output | Use when |
|-----------------|-------|------------|----------|
| `scVI` | Raw counts | `.obsm['X_scVI']` (latent z) | Single or multi-batch integration |
| `scANVI` | Counts + partial labels | `.obs['predicted']` + latent z | Label transfer from reference |
| `totalVI` | RNA + protein counts | Denoised both modalities | CITE-seq |
| `PEAKVI` | ATAC fragment counts | Latent ATAC space | scATAC-seq |

**Key API pattern:**
```python
Model.setup_anndata(adata, layer='counts', batch_key='batch', ...)
model = Model(adata, ...)
model.train(max_epochs=...)
adata.obsm['X_Model'] = model.get_latent_representation()
```

**Next:** T02 — Squidpy for spatial transcriptomics analysis.